In [25]:
import os
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [27]:
from langgraph.checkpoint.memory import MemorySaver,InMemorySaver

In [29]:
from data.marvel import marvel_action_figures
from data.dc import dc_action_figures
from data.specs import specs


In [30]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")


In [31]:
# Create Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.6,
    google_api_key=api_key
)

# Test the model
response = llm.invoke("What is Iron Man's real name?")

print(response.text)

Iron Man's real name is **Tony Stark**.


In [32]:


action_figures = {
    **marvel_action_figures,
    **dc_action_figures
}




In [33]:
@tool
def get_actionfigure_details(af_name: str):

    """Get all details about an action figure from given name"""

    af_info=action_figures.get(af_name)

    if af_info:
        figure = action_figures[af_name]

        return {
            "name": af_name,
            "price_lkr": figure["price_lkr"],
            "quantity": figure["quantity"],
            "description": figure["description"]
        }
    else:
        return f"{af_name} Figure not found"



@tool
def get_af_specs(af_name: str):
    """Get specifications about a specific action figure by name."""

    spec_info = specs.get(af_name)

    if spec_info:
        return spec_info
    else:
        return f"No specs found for {af_name}"


@tool
def write_to_file(content:str,filename:str):
    """Write content to a file given it's name"""

    try:
        with open(filename,'w') as file:
            file.write(content)
        return f"Content written to {filename} successfully."
    except Exception as e:
        print (f"eRROR:{E}")
        return f"Error writing :{e}"       


@tool
def read_file(filename:str):
    """Read content from a file given it's name"""     
    if os.path.exists(filename):
        try:
            with open(filename,'r') as file:
                return file.read()

        except Exception as e:
            print (f"eRROR:{E}")
            return f"Error reading :{e}" 

        
    else:
        return f"File {filename} doesn't exist"          

In [17]:
system_prompt = """   

You are a helpful AI assistant for a Marvel and DC action figure store.

You can use these tools:

- get_actionfigure_details: Get an action figure's name, price, quantity, and description.
- get_af_specs: Get an action figure's powers and accessories.
- write_to_file: Save information to a file when the user asks.
- read_file: Read information from a file when the user asks.

Rules:
- Always use the tools to get product information.
- Never make up prices, stock, descriptions, powers, or accessories.
- Prices are in LKR.
- If a figure is not found, tell the user.
- If quantity is 0, tell the user it is out of stock.
- Be friendly 

"""

#  giving memory to the agent so it can remember previous interactions 

config= {'configurable': {'thread_id':'agent_action'}}

agent = create_agent(
    model=llm,
    tools=[get_actionfigure_details, get_af_specs, write_to_file, read_file],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [34]:
# def use_ac_agent(user_input:str):
#     response = agent.invoke({'messages': [{'role': 'user', 'content': user_input}]},config=config)
#     print(f'Agent RESPONSE:{['messages'][-1].content}')


def use_ac_agent(user_input: str):
    response = agent.invoke(
        {'messages': [{'role': 'user', 'content': user_input}]},
        config=config
    )

    print(f"Agent RESPONSE: {response['messages'][-1].content}")


In [35]:
# use_ac_agent("What is the price of Loki")
use_ac_agent("What is the price of Captain America? and read the detail in a file called 'CAP.txt'")

Agent RESPONSE: The details for Captain America from 'CAP.txt' are: "Name: Captain America, Price: 9000 LKR, Quantity: 20, Description: Captain America action figure with his iconic shield and blue superhero suit."
